# Statistics for International Commerce
## Week 6–7 Illustration Notebook: Sampling Distributions and Confidence Intervals

This notebook is for **illustration and intuition**.

Topics:
- population vs sample
- sampling distribution of the sample mean
- Central Limit Theorem
- standard error
- confidence intervals
- z versus t

Use it in class to **show the ideas visually**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, t

plt.rcParams["figure.dpi"] = 140
plt.rcParams["figure.figsize"] = (8, 4.5)
np.random.seed(42)

## 1. A tiny population

Suppose a small online store has these four order values (USD):

6, 7, 8, 10

We can compute:
- the population mean
- all possible samples of size 2
- the sampling distribution of the sample mean

In [ ]:
from itertools import combinations

population = np.array([6, 7, 8, 10])
population_mean = population.mean()
population_sd = population.std(ddof=0)

samples = list(combinations(population, 2))
sample_means = np.array([np.mean(s) for s in samples])

population_mean, population_sd, samples, sample_means

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(population, bins=np.arange(5.5, 10.6, 1), edgecolor="white")
axes[0].axvline(population_mean, color="red", linestyle="--", label=f"Mean = {population_mean:.2f}")
axes[0].set_title("Population Values")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(sample_means, bins=np.arange(6.25, 9.26, 0.5), edgecolor="white")
axes[1].axvline(sample_means.mean(), color="red", linestyle="--", label=f"Mean = {sample_means.mean():.2f}")
axes[1].set_title("Sampling Distribution of Sample Means")
axes[1].set_xlabel("Sample mean")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Standard error

For the sample mean:

SE = population standard deviation / sqrt(n)

Notice:
- larger sample size -> smaller standard error
- sample means vary less than individual observations

In [ ]:
for n in [4, 9, 16, 25, 64, 100]:
    se = 20 / np.sqrt(n)
    print(f"n = {n:>3}, standard error = {se:.2f}")

## 3. Central Limit Theorem

We now simulate a **skewed population**, then repeatedly draw samples and compute sample means.

Even though the population is skewed, the distribution of the sample mean becomes more normal as sample size increases.

In [ ]:
population_skewed = np.random.exponential(scale=5, size=100000)

sample_means_5 = np.array([np.mean(np.random.choice(population_skewed, 5)) for _ in range(2000)])
sample_means_30 = np.array([np.mean(np.random.choice(population_skewed, 30)) for _ in range(2000)])
sample_means_100 = np.array([np.mean(np.random.choice(population_skewed, 100)) for _ in range(2000)])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0,0].hist(population_skewed[:5000], bins=40, edgecolor="white")
axes[0,0].set_title("Population (skewed)")
axes[0,0].set_xlabel("Value")

axes[0,1].hist(sample_means_5, bins=35, edgecolor="white")
axes[0,1].set_title("Sample Means, n = 5")
axes[0,1].set_xlabel("Sample mean")

axes[1,0].hist(sample_means_30, bins=35, edgecolor="white")
axes[1,0].set_title("Sample Means, n = 30")
axes[1,0].set_xlabel("Sample mean")

axes[1,1].hist(sample_means_100, bins=35, edgecolor="white")
axes[1,1].set_title("Sample Means, n = 100")
axes[1,1].set_xlabel("Sample mean")

plt.tight_layout()
plt.show()

## 4. Business example: export container weights

Suppose export container weight has:
- population mean = 500 kg
- population standard deviation = 20 kg

For a sample of n = 25 containers, what happens to the sampling distribution of the sample mean?

In [ ]:
mu = 500
sigma = 20
n = 25
se = sigma / np.sqrt(n)

x = np.linspace(480, 520, 500)
y = norm.pdf(x, loc=mu, scale=se)

plt.plot(x, y, linewidth=2)
plt.axvline(mu, color="red", linestyle="--", label=f"Mean = {mu}")
plt.title("Sampling Distribution of Sample Mean (Container Weight)")
plt.xlabel("Sample mean weight")
plt.ylabel("Density")
plt.legend()
plt.show()

print(f"Standard error = {se:.2f}")

## 5. Confidence intervals with known sigma

A company samples 64 orders and finds:
- sample mean = 82 USD
- population standard deviation = 16 USD

Construct a 95% confidence interval.

In [ ]:
xbar = 82
sigma = 16
n = 64
z_star = 1.96

se = sigma / np.sqrt(n)
me = z_star * se
lower = xbar - me
upper = xbar + me

print(f"Standard error = {se:.2f}")
print(f"Margin of error = {me:.2f}")
print(f"95% CI = ({lower:.2f}, {upper:.2f})")

## 6. Why larger n gives a narrower interval

We keep the same:
- sample mean = 82
- sigma = 16
- confidence level = 95%

and only change n.

In [ ]:
results = []
for n in [16, 25, 36, 64, 100, 225]:
    se = 16 / np.sqrt(n)
    me = 1.96 * se
    results.append([n, se, me, 2*me])

pd.DataFrame(results, columns=["n", "SE", "Margin_of_Error", "CI_Width"])

## 7. z vs t

When sigma is unknown, we use the t distribution instead of the z distribution.

For small samples, the t distribution is:
- wider
- flatter
- has heavier tails

In [ ]:
x = np.linspace(-4, 4, 500)

plt.plot(x, norm.pdf(x), label="z distribution", linewidth=2)
plt.plot(x, t.pdf(x, df=5), label="t distribution (df=5)", linewidth=2)
plt.plot(x, t.pdf(x, df=20), label="t distribution (df=20)", linewidth=2)
plt.title("Comparing z and t Distributions")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.show()

## 8. Confidence interval with unknown sigma

Suppose a company samples 16 overseas shipments and finds:
- sample mean = 42 USD
- sample standard deviation = 8 USD

Use a 95% confidence interval with t.

In [ ]:
xbar = 42
s = 8
n = 16
df = n - 1
t_star = t.ppf(0.975, df=df)

se = s / np.sqrt(n)
me = t_star * se
lower = xbar - me
upper = xbar + me

print(f"t* = {t_star:.3f}")
print(f"Standard error = {se:.2f}")
print(f"Margin of error = {me:.2f}")
print(f"95% CI = ({lower:.2f}, {upper:.2f})")

## 9. Final takeaway

Main ideas:
- sample means vary from sample to sample
- that variation is described by the sampling distribution
- standard error measures the spread of sample means
- larger samples give more precise estimates
- confidence intervals combine:
  - estimate
  - uncertainty
  - interpretation